# Tool Use GRPO - Qwen3-8B

**Tinker RL Project — PES University MTech Capstone (Group 6)**

| Field | Value |
|-------|-------|
| **Model** | `Qwen/Qwen3-8B` |
| **Benchmark** | Function Calling / Tool Use (glaive-function-calling-v2) |
| **Method** | GRPO + LoRA rank 32 |
| **Reward** | Binary: 1.0 if correct tool + args, 0.0 otherwise |
| **Training API** | Tinker (cloud GPU) |
| **Environment** | Atropos + custom ToolUseEnv |
| **Assigned to** | Pes F (Druva) |
| **Status** | Ready to run |

In [ ]:
# Install dependencies
!pip install -q atroposlib==0.3.0 \
    'git+https://github.com/thinking-machines-lab/tinker.git' \
    datasets transformers wandb pydantic python-dotenv

In [ ]:
# Clone repo
!git clone https://github.com/pes-llm-research/tinker-rl-lab.git
%cd tinker-rl-lab/atropos
!pip install -e . -q

In [ ]:
import os

# Set credentials (use Colab secrets or paste directly)
os.environ['TINKER_API_KEY'] = 'YOUR_TINKER_API_KEY'
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN'
os.environ['TINKER_CONFIG_PATH'] = 'configs/tool_use_qwen_8b.yaml'

print('Credentials set.')

In [ ]:
# Write config
config_yaml = """
env:
  group_size: 16
  batch_size: 128
  max_batches_offpolicy: 3
  tokenizer_name: "Qwen/Qwen3-8B"
  use_wandb: false
  rollout_server_url: "http://localhost:8000"
  wandb_name: "tool-use-qwen3-8b-env"
  ensure_scores_are_not_same: false
  max_token_length: 512
  max_num_workers: 24
  total_steps: 50
  steps_per_eval: 10

openai:
  - model_name: "Qwen/Qwen3-8B"
    base_url: "http://localhost:8001/v1"
    api_key: "x"
    weight: 1.0
    num_requests_for_eval: 200

tinker:
  lora_rank: 32
  learning_rate: 0.00004
  max_token_trainer_length: 1024
  checkpoint_dir: "./checkpoints/tool_use_qwen8b/"
  save_checkpoint_interval: 10
  wandb_project: "tinker-rl-scaling"
  wandb_group: "tool-use-scaling"
  wandb_run_name: "tool-use-qwen3-8b"

slurm: false
testing: false
"""
with open('configs/tool_use_qwen_8b.yaml', 'w') as f:
    f.write(config_yaml)
print('Config written.')

In [ ]:
# Launch Atropos coordinator (background)
!nohup python -m atroposlib.server > /tmp/atropos_server.log 2>&1 &
import time; time.sleep(5)
print('Atropos coordinator started')

In [ ]:
# Launch environment (background)
!nohup python -m tinker_atropos.environments.tool_use_tinker \
    --config configs/tool_use_qwen_8b.yaml \
    > /tmp/tool_use_env.log 2>&1 &
import time; time.sleep(10)
print('Environment started')
!tail -5 /tmp/tool_use_env.log

In [ ]:
# Launch trainer (foreground — this runs the GRPO training loop)
!python launch_training.py --config configs/tool_use_qwen_8b.yaml

In [ ]:
# RESULTS — paste step data here after training completes
# Format: steps = [...], rewards = [...]

steps = []   # TODO: fill in after training
rewards = [] # TODO: fill in after training

if steps and rewards:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(steps, rewards, color='#9b59b6', linewidth=2)
    ax.fill_between(steps, rewards, alpha=0.15, color='#9b59b6')
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Tool Call Accuracy')
    ax.set_title('Tool Use GRPO - Qwen3-8B\nReward Trajectory')
    ax.set_ylim(-0.05, 1.1)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f'Initial: {rewards[0]:.4f} | Final: {rewards[-1]:.4f}')
else:
    print('Run training first, then paste step/reward data here.')